In [ ]:
import pandas as pd
import os
from transformers import AutoTokenizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
FINE_TUNE_LEN = 128
ARCH_LEN = 512
TOKENIZER = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")


def get_tokens(df):
    print("Calculating token lengths...")
    return TOKENIZER(df["text"].tolist(), truncation=False, add_special_tokens=False)


def analyze_token_statistics(df):
    tokens = get_tokens(df)

    token_counts = pd.Series([len(ids) for ids in tokens["input_ids"]])

    print("\n--- Token Statistics ---")
    print(token_counts.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]))

    over_fine_tune_len = (token_counts > FINE_TUNE_LEN).sum()
    over_arch_len = (token_counts > ARCH_LEN).sum()

    pct_over_fine_tune_len = (over_fine_tune_len / len(df)) * 100
    pct_over_arch_len = (over_arch_len / len(df)) * 100

    print(f"\nModel training limit: {FINE_TUNE_LEN} tokens")
    print(f"Texts over the limit: {over_fine_tune_len} of {len(df)} ({pct_over_fine_tune_len:.2f}%)")

    print(f"\nArchitecture limit: {ARCH_LEN} tokens")
    print(f"Texts over the limit: {over_arch_len} of {len(df)} ({pct_over_arch_len:.2f}%)")


def print_class_distribution(df, parent_col=None):
    print(f"Total number of classes: {df['label_col'].nunique()}\n")
    
    if parent_col and parent_col in df.columns:
        dist = df.groupby([parent_col, 'label_col']).size().reset_index(name='samples')
        print(dist.to_string(index=False))
    else:
        dist = df['label_col'].value_counts().reset_index()
        dist.columns = ['class', 'samples']
        print(dist.to_string(index=False))

In [ ]:

data_path = "../data/WebOfScience/WOS11967"

with open(os.path.join(data_path, "X.txt"), "r", encoding="utf-8") as f:
    texts = [line.strip() for line in f.readlines()]

with open(os.path.join(data_path, "Y.txt"), "r", encoding="utf-8") as f:
    y = [int(line.strip()) for line in f.readlines()]
    
with open(os.path.join(data_path, "YL1.txt"), "r", encoding="utf-8") as f:
    yl1 = [int(line.strip()) for line in f.readlines()]
    
with open(os.path.join(data_path, "YL2.txt"), "r", encoding="utf-8") as f:
    yl2 = [int(line.strip()) for line in f.readlines()]

df = pd.DataFrame({
    "text": texts,
    "label_col": y,
    "label_l1": yl1,
    "label_l2": yl2
})

print(f"Successfully loaded dataset from {data_path}. Number of samples: {len(df)}")

analyze_token_statistics(df)
print()
print_class_distribution(df, "label_l1")

In [ ]:
from lingua import Language, LanguageDetectorBuilder

LANG_DETECTOR = LanguageDetectorBuilder.from_all_languages().build()


def is_english(
    text: str,
    english_threshold: float = 0.80,
    non_english_threshold: float = 0.90,
    min_chars: int = 30,
) -> bool:
    if len(text) < min_chars:
        return True

    try:
        confidence_values = LANG_DETECTOR.compute_language_confidence_values(text[:1000])
    except Exception:
        return True

    if not confidence_values:
        return True

    top = confidence_values[0]
    en_score = 0.0
    for value in confidence_values:
        if value.language == Language.ENGLISH:
            en_score = value.value
            break

    if en_score >= english_threshold:
        return True

    if top.language != Language.ENGLISH and top.value >= non_english_threshold:
        return False

    return True


data_path = "../data/Ankaadia/data_2025_10_26.csv"
df = pd.read_csv(data_path)

df = df[['content_en', 'label']].rename(columns={'content_en': 'text', 'label': 'label_col'})
df = df.drop_duplicates(subset=["text"])

print(f"Successfully loaded dataset from {data_path}.\n")

tokens = get_tokens(df)

df['token_count'] = [len(ids) for ids in tokens["input_ids"]]

# Look at very long and short documents to find a max and min token limit beyond which the text is unusable
if not os.path.exists("output"):
    os.makedirs("output")
q01 = df['token_count'].quantile(0.01)
q01_samples = df[df['token_count'] <= q01].sort_values(by='token_count', ascending=True)
q01_samples.to_csv("output/q01_token_samples.csv", sep=";", index=False, encoding="utf-8")
q99 = df['token_count'].quantile(0.99)
q99_samples = df[df['token_count'] >= q99].sort_values(by='token_count', ascending=False)
q99_samples.to_csv("output/q99_token_samples.csv", sep=";", index=False, encoding="utf-8")

df['lang'] = df['text'].apply(is_english)

min_tokens_limit = 15
# max_tokens_limit = 50000

short_mask = df['token_count'] < min_tokens_limit
#long_mask = df['token_count'] > max_tokens_limit
lang_mask = df['lang'] == False

bad_mask = lang_mask | short_mask # | long_mask

print(f"\nTotal number of samples: {len(df)} ")
print_class_distribution(df)
print()
analyze_token_statistics(df)

print(f"\nToo short: {short_mask.sum()} (< {min_tokens_limit} tokens)")
# print_class_distribution(df[~short_mask])
# print(f"\nToo long: {long_mask.sum()} (> {max_tokens_limit} tokens)")
# print_class_distribution(df[~long_mask])
print(f"\nNon-English: {lang_mask.sum()}")
# print_class_distribution(df[~lang_mask])
df_clean = df[~bad_mask].copy()
print(f"\nNumber of non-english or bad samples removed: {bad_mask.sum()}")

print_class_distribution(df_clean)
print()
analyze_token_statistics(df_clean)